# Tencent WeMM-Embedding-9B + Qdrant — Kaggle T4×2 Production Demo

## Steps 1/8–5/8 — Bootstrap + chuẩn bị hệ thống / Bootstrap + system setup

**VI:** Cell đầu tiên checkout presentation source từ public release `v1.0.0`, bootstrap frozen science/runtime authority tại commit `d04bcd3e601b449b67d09ff1132cab965619d858`, cài `requirements-kaggle.txt` + `requirements-demo.txt`, xác minh T4×2/Kaggle Inputs, reuse hoặc restore Qdrant và load GPU worker.

**EN:** The first code cell checks out presentation source from public release `v1.0.0`, bootstraps frozen science/runtime authority at commit `d04bcd3e601b449b67d09ff1132cab965619d858`, installs `requirements-kaggle.txt` + `requirements-demo.txt`, verifies T4×2/Kaggle Inputs, reuses or restores Qdrant, and loads the GPU worker.

**Runtime requirements:** T4 ×2 · Internet ON · Dataset `dangkhoa2016/wemm-embedding-9b-v1-qdrant-snapshots` v1 · Model `dangkhoa2016/tencent-wemm-embedding-9b` v1.

In [ ]:
from pathlib import Path
import importlib
import shutil
import subprocess
import sys

REPO = "https://github.com/dangkhoa2016/Tencent-WeMM-Embedding-9B-Kaggle-T4x2-GPU.git"
PUBLIC_RELEASE_REF = "v1.0.0"
SOURCE_ROOT = Path("/kaggle/working/wemm-public-notebook-source-v1.0.0")

if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir(parents=True)
subprocess.run(["git", "init", "-q"], cwd=SOURCE_ROOT, check=True)
subprocess.run(["git", "remote", "add", "origin", REPO], cwd=SOURCE_ROOT, check=True)
subprocess.run(["git", "fetch", "-q", "--depth", "1", "origin", PUBLIC_RELEASE_REF], cwd=SOURCE_ROOT, check=True)
subprocess.run(["git", "checkout", "-q", "--detach", "FETCH_HEAD"], cwd=SOURCE_ROOT, check=True)

sys.path.insert(0, str(SOURCE_ROOT))
importlib.invalidate_caches()
import wemm_notebook
from wemm_notebook import start_public_session, run_step6, run_step7a, run_step7b, run_step8

presentation_file = Path(wemm_notebook.__file__).resolve()
assert SOURCE_ROOT.resolve() in presentation_file.parents, (presentation_file, SOURCE_ROOT)
print("PUBLIC_NOTEBOOK_PRESENTATION_SOURCE=PASS", flush=True)
print("PUBLIC_NOTEBOOK_PRESENTATION_REF=" + PUBLIC_RELEASE_REF, flush=True)

demo = start_public_session()

## Step 6/8 — Truy xuất văn bản song ngữ / Bilingual text retrieval

**VI:** Giữ nguyên 5 frozen EN↔VI examples và 20 retrieval paths. Hiển thị full query text, full candidate text, TOP-1 winner và nearest competitor; không clipping và raw cosine không được diễn giải thành confidence percentage.

**EN:** Preserve the five frozen EN↔VI examples and 20 retrieval paths. Show full query text, full candidate text, the TOP-1 winner, and the nearest competitor; no clipping and raw cosine is never presented as a confidence percentage.

In [ ]:
text_results = run_step6(demo)

## Step 7A/8 — Truy xuất semantic ảnh→văn bản / Semantic image→text retrieval

**VI:** 4 frozen image→text examples chạy trên production Qdrant corpus `99,967` entities mỗi collection, ở cả `4096d` và `1024d`. Đây là semantic corpus retrieval và được báo cáo riêng với Step 7B.

**EN:** Four frozen image→text examples run over the production Qdrant corpus of `99,967` entities per collection at both `4096d` and `1024d`. This is semantic corpus retrieval and is reported separately from Step 7B.

In [ ]:
image_results = run_step7a(demo)

## Step 7B/8 — Độ bền truy xuất hình ảnh / Visual robustness retrieval

**VI:** Search space là temporary gallery đúng 4 original images. Mỗi entity chạy 4 transforms × 2 dimensions = 8 paths, tổng `32` paths. PASS yêu cầu `32/32` rank #1 với raw cosine `>= 0.90`; không rescale, không nới threshold.

**EN:** The search space is a temporary gallery containing exactly four original images. Each entity runs 4 transforms × 2 dimensions = 8 paths, `32` paths total. PASS requires `32/32` rank #1 with raw cosine `>= 0.90`; no rescaling and no threshold relaxation.

In [ ]:
visual_results = run_step7b(demo)

## Step 8/8 — Đóng phiên + nghiệm thu / Closeout + acceptance

**VI:** Closeout giải phóng GPU worker, xác minh VRAM reclaim, dừng và seal Qdrant. Scorecard cuối giữ tách biệt semantic `36/36 TOP-1` trên corpus `99,967` entities và visual robustness `32/32 TOP-1` trên gallery 4 ảnh. `68/68` chỉ là **TOTAL EXECUTED RETRIEVAL CHECKS**.

**EN:** Closeout releases the GPU worker, verifies VRAM reclaim, stops and seals Qdrant. The final scorecard keeps semantic `36/36 TOP-1` over the `99,967`-entity corpus separate from visual robustness `32/32 TOP-1` over the four-image gallery. `68/68` is only **TOTAL EXECUTED RETRIEVAL CHECKS**.

In [ ]:
final_summary = run_step8(demo, visual_results)